In [139]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

from retrieval_pipeline import *

cfg = load_config()

In [140]:
import importlib
import retrieval_pipeline
importlib.reload(retrieval_pipeline)
from retrieval_pipeline import *
cfg = load_config()
cfg['hf_model']

'deepseek-ai/DeepSeek-V4-Pro'

In [141]:
cfg['hf_model']

'deepseek-ai/DeepSeek-V4-Pro'

In [142]:
vstore = get_vector_store(cfg)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [143]:
client = InferenceClient(api_key=cfg['hf_token'] if cfg.get('hf_token') else None)

In [144]:
sample_query = 'How do i integrate slack with clouddesk?'
# Avoid unsupported CUDA kernels on this device
embedding_model = vstore[2]
embedding_model.to("cpu")

# Ensure retrieval uses the CPU-backed embedding model
vstore = (vstore[0], vstore[1], embedding_model)

docs = retrieve_docs(vstore, sample_query, k=3)
context_str, citation_str = format_docs(docs)

In [145]:
context_str

"[Hc-101 Integrating-Clouddesk-With-Slack (help_center_articles)]\nIntegrating CloudDesk with Slack\nDoc ID: HC-101  |  Source type: help_center_article  |  Title: Integrating CloudDesk with Slack  |  Last updated: 2026-06-18  | \nVersion: v3.6\nCloudDesk can post real-time notifications to Slack whenever a ticket is created, updated, or replied to —\nkeeping your team in the loop without needing to check the dashboard.\nStep 1: Connect your Slack workspace\nGo to Settings → Integrations → Slack → Connect. You'll be redirected to Slack to authorize CloudDesk.\nApprove\n\n---\n\n[Ticket tkt_10021: Slack messages not posting to channel (support_ticket)]\nTicket [tkt_10021] Subject: Slack messages not posting to channel\nQuestion: We connected CloudDesk to Slack but new tickets aren't showing up in #support-alerts.\nResolution: Confirmed the OAuth token had an expired scope after a Slack workspace admin change. Reconnected the integration under Settings > Integrations > Slack and re-selec

In [146]:
citation_str

'- **[Hc-101 Integrating-Clouddesk-With-Slack]** (`help_center_articles`) — Similarity: 75%\n- **[Ticket tkt_10021: Slack messages not posting to channel]** (`support_ticket`) — Similarity: 58%\n- **[Authentication]** (`api_documentation`) — Similarity: 55%'

In [147]:
sample_query = 'How do i integrate slack with clouddesk?'
result = run_rag_pipeline(cfg, vstore, client, sample_query)

In [148]:
sample_query = 'How do i make a cake?'
result = run_rag_pipeline(cfg, vstore, client, sample_query)

In [149]:
print(f'Query: {sample_query}')
print(f"Confidence: {result['confidence_pct']}")
print(f"Escalated: {result['requires_escalation']}\n")

print('------- Answer Output -------')
print(result['answer'])

Query: How do i make a cake?
Confidence: 10%
Escalated: True

------- Answer Output -------
I'm sorry, but I cannot answer your question about how to make a cake based on the provided context.

The context I have available only contains information about the CloudDesk REST API, including endpoints for managing customers and tickets, and details on API key authentication. There is no information related to baking or recipes.

For questions outside the scope of the CloudDesk API, please contact Tier-2 Support for further assistance.


In [150]:
metrics = evaluate_benchmark(cfg, vstore, client)
df_results = pd.DataFrame(metrics['records'])

display(df_results)

[*] Running RAG Evaluation Benchmark across 43 historical tickets...
[*] HuggingFace API notice: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ab81548-056828b248ea00147bd4a95a;1fe1ca67-2592-4dcf-ace7-e0bfd7d70af2)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
[*] HuggingFace API notice: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ab81549-57fc7d580be075935c3a0d6c;fabed838-3ada-4967-bf62-b3787965ee1c)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to

,ticket_id,question,ref_title,top_retrieved,confidence,escalated,hit_at_1,hit_at_3
0,tkt_10021,We connected CloudDesk to Slack but new ticket...,Integrating CloudDesk with Slack,Ticket tkt_10021: Slack messages not posting t...,0.78,False,False,False
1,tkt_10022,Can different ticket categories go to differen...,Integrating CloudDesk with Slack,Ticket tkt_10022: How do I map tickets to spec...,0.75,False,False,False
2,tkt_10023,"After enabling SSO for our workspace, the Slac...",Integrating CloudDesk with Slack,Ticket tkt_10023: Slack app shows disconnected...,0.79,False,False,False
3,tkt_10031,Our webhook receiver is getting events hours a...,Webhooks Reference,Ticket tkt_10031: Webhooks arriving 2+ hours late,0.74,False,False,False
4,tkt_10032,Every webhook we receive fails HMAC signature ...,Webhooks Reference,Ticket tkt_10032: Signature verification alway...,0.83,False,False,False
5,tkt_10033,Our webhook endpoint stopped receiving anythin...,Webhook Delivery Failure Response,Ticket tkt_10033: Webhook got auto-disabled,0.75,False,False,True
6,tkt_10034,We added a second domain to our workspace last...,SSO/SAML Authentication Outage Response,Ticket tkt_10034: My SAML login stopped workin...,0.72,False,False,True
7,tkt_10041,Getting 'saml_signature_invalid' for all users...,SAML & SCIM Configuration Reference,Ticket tkt_10041: SAML signature invalid error,0.69,False,False,True
8,tkt_10042,We verified our new domain but users on it can...,SAML & SCIM Configuration Reference,Ticket tkt_10042: New domain not enforcing SSO,0.64,False,False,False
9,tkt_10043,Provisioning a new user via SCIM throws scim_d...,SAML & SCIM Configuration Reference,Runbook: SSO/SAML Authentication Outage Response,0.81,False,False,True
